# LivePortrait 宠物网格图生成器 (Google Colab)

本 Notebook 用于在 Colab 上运行 LivePortrait（动物版）生成宠物多视角网格图，供前端交互使用。

## 工作流程：
1. 安装依赖和下载模型
2. 上传宠物照片和驱动视频
3. 运行 LivePortrait 生成动态视频
4. 从视频中提取帧并拼接成网格图（7x7 或 11x11）
5. 下载网格图到本地


## 步骤 1: 检查 GPU 并安装依赖


In [ ]:
# 检查 GPU 是否可用
import torch
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ 警告: 未检测到 GPU，请确保 Colab 已分配 GPU 运行时（运行时 -> 更改运行时类型 -> GPU）")


In [ ]:
# 安装基础依赖
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python pillow numpy imageio imageio-ffmpeg
!pip install -q face-alignment dlib
!pip install -q onnxruntime onnxruntime-gpu
!pip install -q insightface

# 安装 LivePortrait 特定依赖
!pip install -q tyro  # 命令行参数解析
!pip install -q einops  # 张量操作
!pip install -q timm  # 预训练模型
!pip install -q diffusers  # 扩散模型（如果需要）
!pip install -q transformers  # Transformers 模型

print("✅ 依赖安装完成")


## 步骤 2: 克隆 LivePortrait 仓库并下载模型


In [ ]:
# 克隆 LivePortrait 官方仓库
import os
if not os.path.exists('LivePortrait'):
    !git clone https://github.com/KwaiVGI/LivePortrait.git
    print("✅ LivePortrait 仓库克隆完成")
else:
    print("✅ LivePortrait 仓库已存在")

os.chdir('LivePortrait')
print(f"当前目录: {os.getcwd()}")

# 安装 LivePortrait 的依赖（如果有 requirements.txt）
if os.path.exists("requirements.txt"):
    print("\n📦 发现 requirements.txt，安装 LivePortrait 依赖...")
    !pip install -q -r requirements.txt
    print("✅ LivePortrait 依赖安装完成")
else:
    print("\n⚠️ 未找到 requirements.txt，将使用通用依赖")


## 步骤 2.1: 下载官方预训练权重（HuggingFace）

LivePortrait 代码默认从 `pretrained_weights/` 目录加载 `.pth` 权重。执行下方命令拉取官方完整权重（包含 base_models）。


In [ ]:
# 下载官方预训练权重（需 git-lfs）
!git lfs install

# 克隆官方权重到临时目录
!git clone https://huggingface.co/KwaiVGI/LivePortrait temp_pretrained_weights

# 创建目标目录并移动权重
!mkdir -p pretrained_weights
!mv temp_pretrained_weights/* pretrained_weights/
!rm -rf temp_pretrained_weights

# 查看关键权重是否就位
print("\n" + "=" * 60)
print("📁 预训练权重目录预览")
print("=" * 60)
!ls -lh pretrained_weights | head

print("\n子目录 liveportrait/base_models 预览:")
!ls -lh pretrained_weights/liveportrait/base_models | head


In [ ]:
# 创建模型目录
!mkdir -p checkpoints pretrained_weights

# 安装 huggingface_hub 用于下载模型
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
import gdown

print("=" * 60)
print("📥 下载 LivePortrait 模型")
print("=" * 60)

# 方法 1: 从 HuggingFace 下载（推荐，包含动物模型）
print("\n方法 1: 从 HuggingFace 下载（推荐）")
print("来源: https://huggingface.co/Kijai/LivePortrait_safetensors")

try:
    from huggingface_hub import list_repo_files, hf_hub_download
    
    # 先列出仓库中的所有文件
    print("正在检查 HuggingFace 仓库中的文件...")
    repo_files = list_repo_files(repo_id="Kijai/LivePortrait_safetensors", repo_type="model")
    print(f"找到 {len(repo_files)} 个文件:")
    for f in repo_files:
        print(f"  - {f}")
    
    # 下载所有 .safetensors 或 .pth 文件
    model_files = [f for f in repo_files if f.endswith(('.safetensors', '.pth', '.ckpt'))]
    
    if model_files:
        print(f"\n正在下载 {len(model_files)} 个模型文件...")
        for model_file in model_files:
            try:
                print(f"  下载: {model_file}")
                hf_hub_download(
                    repo_id="Kijai/LivePortrait_safetensors",
                    filename=model_file,
                    local_dir="checkpoints",
                )
                print(f"  ✅ {model_file} 下载完成")
            except Exception as e:
                print(f"  ⚠️ {model_file} 下载失败: {e}")
    else:
        print("⚠️ 未找到模型文件（.safetensors, .pth, .ckpt）")
        print("   仓库可能为空或文件格式不同")
        
except Exception as e:
    print(f"⚠️ HuggingFace 访问失败: {e}")
    print("   可能原因:")
    print("   1. 仓库不存在或已更名")
    print("   2. 需要认证（设置 HF_TOKEN）")
    print("   3. 网络问题")
    print("   尝试其他下载方法...")

# 方法 2: 从官方 GitHub 仓库下载
print("\n" + "=" * 60)
print("方法 2: 从官方 GitHub 仓库下载")
print("=" * 60)

# 检查官方仓库中是否有 pretrained_weights
if os.path.exists("pretrained_weights"):
    print("检查 pretrained_weights 目录...")
    !ls -la pretrained_weights/ 2>/dev/null || echo "目录为空或不存在"
    
    # 尝试从官方仓库下载（如果有提供下载脚本）
    if os.path.exists("download_weights.sh"):
        print("找到下载脚本，执行中...")
        !bash download_weights.sh
    elif os.path.exists("scripts/download_weights.sh"):
        print("找到下载脚本，执行中...")
        !bash scripts/download_weights.sh
    else:
        print("⚠️ 未找到自动下载脚本")
        print("请访问官方仓库查看下载说明:")
        print("https://github.com/KwaiVGI/LivePortrait")
        print("\n💡 提示: 官方仓库的 README 中通常有 Google Drive 或百度网盘下载链接")
        print("   您可以:")
        print("   1. 查看 README.md 文件中的 Model Zoo 部分")
        print("   2. 使用 gdown 从 Google Drive 下载（如果有链接）")
        print("   3. 手动下载后上传到 Colab")

# 方法 3: 使用 Google Drive 下载（如果有链接）
print("\n" + "=" * 60)
print("方法 3: 使用 Google Drive 下载（如果有官方链接）")
print("=" * 60)

# 检查 README 中是否有 Google Drive 链接
readme_path = "README.md"
if os.path.exists(readme_path):
    print("检查 README.md 中是否有下载链接...")
    with open(readme_path, 'r', encoding='utf-8', errors='ignore') as f:
        readme_content = f.read()
        if 'drive.google.com' in readme_content or 'gdown' in readme_content:
            print("✅ 发现 Google Drive 链接，请查看 README.md 获取下载命令")
        else:
            print("⚠️ README 中未找到 Google Drive 链接")
else:
    print("⚠️ 未找到 README.md 文件")

# 方法 4: 手动下载提示
print("\n" + "=" * 60)
print("方法 4: 手动下载（如果自动下载失败）")
print("=" * 60)
print("""
如果自动下载失败，您可以手动下载：

1. **HuggingFace**:
   - 访问: https://huggingface.co/Kijai/LivePortrait_safetensors
   - 手动浏览文件列表，下载所有模型文件
   - 上传到 Colab 的 checkpoints/ 目录

2. **官方 GitHub**:
   - 访问: https://github.com/KwaiVGI/LivePortrait
   - 查看 README 中的 Model Zoo 部分
   - 通常有 Google Drive 或百度网盘链接
   - 下载后上传到 Colab

3. **社区整合包**（包含动物模型，推荐）:
   - 访问: https://aiyy.info/liveportrait/
   - 下载一键整合包（通常包含动物模型）
   - 解压后，将模型文件上传到 Colab 的 checkpoints/ 目录

4. **在 Colab 中手动上传**:
   - 点击左侧文件图标（📁）
   - 上传下载的模型文件到 checkpoints/ 目录
   - 支持 .pth, .safetensors, .ckpt 格式
""")

# 列出已下载的文件
print("\n" + "=" * 60)
print("📁 当前 checkpoints 目录内容:")
print("=" * 60)
!ls -lh checkpoints/ 2>/dev/null || echo "目录为空"

print("\n" + "=" * 60)
print("📁 当前 pretrained_weights 目录内容:")
print("=" * 60)
!ls -lh pretrained_weights/ 2>/dev/null || echo "目录为空"


In [ ]:
from google.colab import files
import shutil

# 创建输入目录
!mkdir -p inputs

print("📤 请上传您的宠物照片（建议: 正脸、清晰、无遮挡）")
uploaded = files.upload()

# 将上传的图片移动到 inputs 目录
for filename in uploaded.keys():
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        shutil.move(filename, f"inputs/pet_source.jpg")
        print(f"✅ 宠物照片已保存: inputs/pet_source.jpg")
        break


In [ ]:
print("📤 请上传驱动视频（头部转动的视频，可以是人脸或动物的头部转动）")
print("💡 提示: 如果您已有 driving_video.mp4，可以直接上传")
uploaded_video = files.upload()

for filename in uploaded_video.keys():
    if filename.lower().endswith(('.mp4', '.mov', '.avi')):
        shutil.move(filename, f"inputs/driving_video.mp4")
        print(f"✅ 驱动视频已保存: inputs/driving_video.mp4")
        break


## 步骤 4: 运行 LivePortrait 推理（生成动态视频）

**注意**: LivePortrait 的推理命令可能因版本而异，请根据官方文档调整。


## 步骤 3.5: 安装缺失的依赖（如果遇到 ModuleNotFoundError）

如果执行推理时遇到 `ModuleNotFoundError`，运行下面的单元格安装缺失的包：


In [ ]:
# 安装 LivePortrait 可能需要的额外依赖
# 如果遇到 ModuleNotFoundError，运行此单元格

!pip install -q tyro einops timm diffusers transformers
!pip install -q safetensors  # 用于加载 .safetensors 文件
!pip install -q accelerate  # 加速推理

# 如果 LivePortrait 需要其他依赖，查看 requirements.txt
if os.path.exists("requirements.txt"):
    print("发现 requirements.txt，安装其中的依赖...")
    !pip install -q -r requirements.txt
else:
    print("未找到 requirements.txt，已安装常用依赖")

print("✅ 额外依赖安装完成")


In [ ]:
# 检查 LivePortrait 的推理脚本位置
import os
print("当前目录结构:")
!ls -la

# 通常推理脚本在根目录或 scripts/ 目录下
# 根据实际仓库结构调整路径
inference_script = "inference.py"  # 或 "scripts/inference.py" 或其他
if not os.path.exists(inference_script):
    print(f"\n⚠️ 未找到 {inference_script}，请检查 LivePortrait 仓库结构")
    print("可能需要查看官方 README 了解正确的推理命令")


In [ ]:
# 运行推理（使用动物模型）
# 注意：实际参数名可能因版本而异，请根据官方文档调整

source_image = "inputs/pet_source.jpg"
driving_video = "inputs/driving_video.mp4"
output_video = "outputs/pet_animated.mp4"

!mkdir -p outputs

# 检查模型文件是否存在
import os
animal_dir = "checkpoints/animal"
if os.path.exists(animal_dir):
    print("✅ 检测到动物模型目录，将使用动物模型进行推理")
    use_animal = True
else:
    print("⚠️ 未找到动物模型目录，将使用基础模型（对人脸支持更好）")
    use_animal = False

# 检查输入文件
if not os.path.exists(source_image):
    print(f"❌ 错误: 源图片不存在: {source_image}")
    print("   请先执行步骤 3 上传宠物照片")
elif not os.path.exists(driving_video):
    print(f"❌ 错误: 驱动视频不存在: {driving_video}")
    print("   请先执行步骤 3 上传驱动视频")
else:
    print(f"\n📝 准备运行推理:")
    print(f"   源图片: {source_image}")
    print(f"   驱动视频: {driving_video}")
    print(f"   输出视频: {output_video}")
    print(f"   使用模型: {'动物模型' if use_animal else '基础模型'}")
    
    # 根据 LivePortrait 的实际 API 调整以下代码
    # 方法 1: 如果使用命令行接口
    if os.path.exists("inference.py"):
        print("\n🔧 方法 1: 使用命令行接口")
        print("=" * 60)
        
        if use_animal:
            print("使用动物模型，命令示例:")
            print(f"python inference.py --source {source_image} --driving {driving_video} --output {output_video} --checkpoint_dir checkpoints/animal --mode animal")
            print("\n或者尝试:")
            print(f"python inference.py --source {source_image} --driving {driving_video} --output {output_video} --checkpoint_dir checkpoints/animal")
        else:
            print("使用基础模型，命令示例:")
            print(f"python inference.py --source {source_image} --driving {driving_video} --output {output_video} --checkpoint_dir checkpoints")
        
        print("\n" + "=" * 60)
        print("⚠️ 注意: 实际命令可能不同，请查看 LivePortrait 官方文档")
        print("   如果命令失败，请检查:")
        print("   1. inference.py 的实际参数名称（使用 --help 查看）")
        print("   2. 是否需要指定 checkpoint 路径")
        print("   3. 是否需要设置其他参数（如 landmark_model）")
        print("\n💡 提示: 可以先运行以下命令查看帮助:")
        print("   python inference.py --help")
        print("   或")
        print("   python inference.py -h")
        
        # 可选：自动执行命令
        print("\n" + "=" * 60)
        print("🚀 准备执行推理命令...")
        print("=" * 60)
        
        # 注意：以下代码需要根据实际的 inference.py 参数调整
        # 如果 inference.py 的参数不同，请修改下面的命令
        import subprocess
        from subprocess import TimeoutExpired
        
        try:
            if use_animal:
                # 尝试使用动物模型
                # 注意：实际参数名可能不同，请根据 inference.py --help 的输出调整
                print("使用动物模型执行推理...")
                result = subprocess.run([
                    "python", "inference.py",
                    "--source", source_image,
                    "--driving", driving_video,
                    "--output", output_video,
                    "--checkpoint_dir", "checkpoints/animal"
                ], capture_output=True, text=True, timeout=600)
                
                if result.returncode == 0:
                    print("✅ 推理完成！")
                    print(f"输出视频: {output_video}")
                else:
                    print("⚠️ 命令执行失败，可能参数不正确")
                    print("错误信息:")
                    print(result.stderr)
                    print("\n请查看上面的命令示例，手动调整参数后执行")
            else:
                print("使用基础模型执行推理...")
                result = subprocess.run([
                    "python", "inference.py",
                    "--source", source_image,
                    "--driving", driving_video,
                    "--output", output_video,
                    "--checkpoint_dir", "checkpoints"
                ], capture_output=True, text=True, timeout=600)
                
                if result.returncode == 0:
                    print("✅ 推理完成！")
                    print(f"输出视频: {output_video}")
                else:
                    print("⚠️ 命令执行失败，可能参数不正确")
                    print("错误信息:")
                    print(result.stderr)
                    print("\n请查看上面的命令示例，手动调整参数后执行")
        except FileNotFoundError:
            print("⚠️ 未找到 inference.py，请检查 LivePortrait 仓库结构")
        except TimeoutExpired:
            print("⚠️ 推理超时（超过 10 分钟），可能需要更长时间")
        except Exception as e:
            print(f"⚠️ 执行出错: {e}")
            print("   请手动执行上面的命令示例")
        
    # 方法 2: 如果使用 Python API
    elif os.path.exists("liveportrait") or os.path.exists("src"):
        print("\n🔧 方法 2: 使用 Python API")
        print("请参考 LivePortrait 官方文档中的 Python API 示例")
        print("通常类似:")
        print("""
from liveportrait import LivePortrait

# 初始化模型（使用动物模型）
model = LivePortrait(
    checkpoint_dir="checkpoints/animal" if use_animal else "checkpoints",
    landmark_model="checkpoints/landmark_model.pth"
)

# 运行推理
model.inference(
    source_image=source_image,
    driving_video=driving_video,
    output_video=output_video
)
        """)
    else:
        print("\n⚠️ 未找到推理脚本，请检查 LivePortrait 仓库结构")
        print("   可能需要:")
        print("   1. 查看 README.md 了解使用方法")
        print("   2. 查看 examples/ 目录中的示例代码")
        print("   3. 查看官方文档: https://github.com/KwaiVGI/LivePortrait")


## 步骤 4.1: 执行推理命令（使用 ! 前缀）

**重要**: 在 Colab 中执行 shell 命令需要使用 `!` 前缀。

如果上面的自动执行失败，可以手动执行下面的命令（取消注释并运行）：


In [ ]:
# 检查文件并显示执行命令
# 注意：在 Colab 中，要执行 shell 命令，请使用下面的新单元格（使用 ! 前缀）

import os
source_image = "inputs/pet_source.jpg"
driving_video = "inputs/driving_video.mp4"
output_video = "outputs/pet_animated.mp4"

print("=" * 60)
print("📋 文件检查")
print("=" * 60)
print(f"inference.py 存在: {os.path.exists('inference.py')}")
print(f"源图片存在: {os.path.exists(source_image)}")
print(f"驱动视频存在: {os.path.exists(driving_video)}")
print(f"动物模型目录存在: {os.path.exists('checkpoints/animal')}")

if os.path.exists("checkpoints/animal"):
    print("\n" + "=" * 60)
    print("✅ 使用动物模型的命令:")
    print("=" * 60)
    print(f"!python inference.py --source {source_image} --driving {driving_video} --output-dir outputs --no-flag-stitching --flag-relative-motion --flag-do-crop")
else:
    print("\n" + "=" * 60)
    print("✅ 使用基础模型的命令:")
    print("=" * 60)
    print(f"!python inference.py --source {source_image} --driving {driving_video} --output-dir outputs --no-flag-stitching --flag-relative-motion --flag-do-crop")

print("\n💡 提示: 请复制上面的命令到下面的新单元格中执行（使用 ! 前缀）")
print("   或者先运行: !python inference.py --help 查看实际参数")


### 执行推理命令

**重要**: 在 Colab 中执行 shell 命令需要使用 `!` 前缀。

**步骤 1**: 先查看帮助（了解实际参数）:
```python
!python inference.py --help
```

**步骤 2**: 根据上面的输出，执行推理命令（推荐动物模型）:
```python
!python inference.py --source inputs/pet_source.jpg --driving inputs/driving_video.mp4 --output-dir outputs --no-flag-stitching --flag-relative-motion --flag-do-crop
```

**使用基础模型**（如果动物模型不想用）:
```python
!python inference.py --source inputs/pet_source.jpg --driving inputs/driving_video.mp4 --output-dir outputs --no-flag-stitching --flag-relative-motion --flag-do-crop
```

**注意**: 
- 布尔开关用 `--flag-xxx` / `--no-flag-xxx`，不要跟 True/False
- 如果命令失败，先运行 `!python inference.py --help` 查看实际参数
- 根据实际错误信息调整命令
- 需要使用官方 `.pth` 权重（已下载到 pretrained_weights/）


## 步骤 4.2: 配置模型路径并执行推理

根据 `--help` 输出，`inference.py` 没有 `--checkpoint-dir` 参数。模型路径可能通过配置文件设置。我们需要检查并配置模型路径。


In [ ]:
# 检查并配置模型路径
import os

# 检查配置文件
print("=" * 60)
print("🔍 检查模型配置")
print("=" * 60)

config_files = ["config.yaml", "config.py", "src/config.py", "src/config/config.py", "src/config/__init__.py"]
found_config = False
for cfg in config_files:
    if os.path.exists(cfg):
        print(f"✅ 找到配置文件: {cfg}")
        found_config = True
        # 查看配置文件内容（前50行）
        try:
            with open(cfg, 'r') as f:
                lines = f.readlines()[:50]
                print(f"\n配置文件内容预览（前50行）:")
                print("".join(lines))
        except:
            pass
        break

if not found_config:
    print("⚠️ 未找到配置文件")

# 检查 checkpoints 目录结构
print("\n" + "=" * 60)
print("📁 检查模型文件")
print("=" * 60)
print("动物模型目录:")
!ls -lh checkpoints/animal/ 2>/dev/null || echo "目录不存在"
print("\n基础模型文件:")
!ls -lh checkpoints/*.safetensors checkpoints/*.pth 2>/dev/null | head -5 || echo "未找到模型文件"

# 配置使用动物模型
# LivePortrait 可能默认使用 checkpoints/ 目录中的基础模型
# 我们需要让它使用 checkpoints/animal/ 目录中的动物模型

print("\n" + "=" * 60)
print("⚙️ 配置使用动物模型")
print("=" * 60)

# 方法 1: 检查是否有配置文件可以修改
# 方法 2: 将动物模型文件复制到默认位置（临时方案）
# 方法 3: 创建符号链接

# 先尝试方法 2：备份基础模型，使用动物模型
if os.path.exists("checkpoints/animal"):
    print("方案：将动物模型设置为默认模型")
    print("（备份基础模型，使用动物模型）")
    
    # 备份基础模型文件（如果存在）
    import shutil
    backup_dir = "checkpoints/backup_human"
    if not os.path.exists(backup_dir):
        os.makedirs(backup_dir)
        for f in os.listdir("checkpoints"):
            if f.endswith(('.safetensors', '.pth')) and f != 'landmark_model.pth':
                src = f"checkpoints/{f}"
                if os.path.isfile(src):
                    shutil.copy2(src, backup_dir)
                    print(f"  备份: {f}")
    
    # 将动物模型文件复制到 checkpoints/ 根目录
    animal_files = os.listdir("checkpoints/animal")
    for f in animal_files:
        if f.endswith('.safetensors'):
            src = f"checkpoints/animal/{f}"
            dst = f"checkpoints/{f}"
            shutil.copy2(src, dst)
            print(f"  复制动物模型: {f} -> checkpoints/")
    
    print("✅ 动物模型已设置为默认模型")
else:
    print("⚠️ 未找到动物模型目录")

# 使用动物模型执行推理（推荐参数）:
# --flag-stitching False: 对于动物，建议关闭 stitching（根据帮助文档）
# --flag-relative-motion True: 使用相对运动（默认）
# --flag-do-crop True: 裁剪源图像（默认）
print("\n🚀 开始执行推理（使用动物模型）...")
print("=" * 60)

!python inference.py --source inputs/pet_source.jpg --driving inputs/driving_video.mp4 --output-dir outputs --flag-stitching False --flag-relative-motion True --flag-do-crop True

# 查看生成的文件
print("\n" + "=" * 60)
print("📁 输出目录中的文件:")
print("=" * 60)
!ls -lh outputs/


In [ ]:
# 执行推理
# 根据 --help 输出，正确的参数格式

# 创建输出目录
!mkdir -p outputs

# 使用动物模型执行推理
# 注意：--flag-stitching False 对动物推荐（根据帮助文档）
print("\n" + "=" * 60)
print("🚀 开始执行推理")
print("=" * 60)
print("使用参数:")
print("  --source: inputs/pet_source.jpg")
print("  --driving: inputs/driving_video.mp4")
print("  --output-dir: outputs")
print("  --flag-stitching: False (推荐用于动物)")
print("  --flag-relative-motion: True")
print("  --flag-do-crop: True")
print("=" * 60)

!python inference.py --source inputs/pet_source.jpg --driving inputs/driving_video.mp4 --output-dir outputs --flag-stitching False --flag-relative-motion True --flag-do-crop True

# 查看生成的文件
print("\n" + "=" * 60)
print("📁 输出目录中的文件:")
print("=" * 60)
!ls -lh outputs/


In [ ]:
# 步骤 1: 查看 inference.py 的帮助信息（了解实际参数）
!python inference.py --help


In [ ]:
# 步骤 2: 执行推理命令（使用动物模型）
# 注意：根据上面的 --help 输出调整参数名称

# 如果使用动物模型（推荐）:
!python inference.py --source inputs/pet_source.jpg --driving inputs/driving_video.mp4 --output outputs/pet_animated.mp4 --checkpoint_dir checkpoints/animal

# 如果上面的命令失败，尝试不使用 --mode 参数:
# !python inference.py --source inputs/pet_source.jpg --driving inputs/driving_video.mp4 --output outputs/pet_animated.mp4 --checkpoint_dir checkpoints/animal

# 或者使用基础模型:
# !python inference.py --source inputs/pet_source.jpg --driving inputs/driving_video.mp4 --output outputs/pet_animated.mp4 --checkpoint_dir checkpoints


## 步骤 5: 从视频提取帧并生成网格图


In [ ]:
# 安装额外依赖（如果需要）
!pip install -q opencv-python-headless

import cv2
import numpy as np
from PIL import Image
import math
import time
import json

def extract_frames_from_video(video_path, target_count=49):
    """从视频中均匀提取指定数量的帧"""
    print(f"正在从视频提取帧: {video_path}")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise Exception(f"无法打开视频文件: {video_path}")
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"  视频总帧数: {total_frames}, FPS: {fps:.2f}, 目标帧数: {target_count}")
    
    images = []
    
    if total_frames <= 0:
        # Fallback: 逐帧读取
        success = True
        while success:
            success, frame = cap.read()
            if success:
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                images.append(img)
    else:
        # 均匀采样
        indices = np.linspace(0, total_frames - 1, target_count, dtype=int)
        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                images.append(img)
            else:
                print(f"    警告: 无法读取第 {i} 帧")
    
    cap.release()
    
    if len(images) == 0:
        raise Exception("未提取到任何帧")
    
    # 如果提取多了，进行采样
    if len(images) > target_count:
        indices = np.linspace(0, len(images) - 1, target_count, dtype=int)
        images = [images[i] for i in indices]
    
    print(f"✅ 成功提取 {len(images)} 帧")
    return images

def create_grid_from_images(images, step=5, output_path="outputs/grid_pet.webp"):
    """将帧列表拼接成网格图"""
    if not images:
        raise Exception("没有有效的图片用于拼接")
    
    count = len(images)
    grid_size = math.ceil(math.sqrt(count))
    cols = grid_size
    rows = grid_size
    
    single_w, single_h = images[0].size
    grid_w = cols * single_w
    grid_h = rows * single_h
    
    print(f"开始拼接: {cols}x{rows}, 总帧数: {count}, 单图 {single_w}x{single_h}")
    
    grid_img = Image.new('RGB', (grid_w, grid_h))
    
    for index, img in enumerate(images):
        if index >= cols * rows:
            break
        col = index % cols
        row = index // cols
        grid_img.paste(img, (col * single_w, row * single_h))
    
    grid_img.save(output_path, 'WEBP', quality=85)
    print(f"✅ 网格图已保存: {output_path}")
    
    # 保存元数据
    meta = {
        "step": step,
        "rows": rows,
        "cols": cols,
        "created_at": time.time()
    }
    meta_path = output_path.replace('.webp', '.json')
    with open(meta_path, 'w') as f:
        json.dump(meta, f)
    print(f"✅ 元数据已保存: {meta_path}")
    
    return output_path, meta

print("✅ 函数定义完成")


In [ ]:
# 从生成的视频中提取帧并生成网格图
video_path = "outputs/pet_animated.mp4"  # 上一步生成的视频路径

# 选择网格大小：7x7=49 或 11x11=121
GRID_SIZE = 7  # 或 11
target_count = GRID_SIZE * GRID_SIZE

if os.path.exists(video_path):
    print(f"正在处理视频: {video_path}")
    frames = extract_frames_from_video(video_path, target_count)
    
    output_path = f"outputs/grid_pet_{GRID_SIZE}x{GRID_SIZE}.webp"
    grid_path, meta = create_grid_from_images(frames, step=5, output_path=output_path)
    
    print(f"\n🎉 完成！网格图已生成: {grid_path}")
    print(f"   尺寸: {meta['rows']}x{meta['cols']}")
else:
    print(f"⚠️ 视频文件不存在: {video_path}")
    print("请先完成步骤 4 生成视频")


## 步骤 6: 下载网格图到本地


In [ ]:
from google.colab import files

# 下载网格图
grid_file = "outputs/grid_pet_7x7.webp"  # 或 grid_pet_11x11.webp
if os.path.exists(grid_file):
    files.download(grid_file)
    print(f"✅ {grid_file} 已开始下载")
    
    # 同时下载元数据文件
    meta_file = grid_file.replace('.webp', '.json')
    if os.path.exists(meta_file):
        files.download(meta_file)
        print(f"✅ {meta_file} 已开始下载")
else:
    print(f"⚠️ 文件不存在: {grid_file}")


## 步骤 7: 将网格图集成到您的 Flask 项目

下载完成后，将 `grid_pet_7x7.webp` 和 `grid_pet_7x7.json` 复制到您的 `face_demo/outputs/` 目录，
然后在前端页面中加载即可使用。

### 使用方式：
1. 将下载的 `.webp` 文件放到 `outputs/` 目录
2. 在浏览器中访问 `http://localhost:5000`
3. 点击"加载已生成的 Grid 图"，输入路径如 `/outputs/grid_pet_7x7.webp`
4. 或者直接修改 `index.html` 中的默认加载路径


---

## 故障排除

### 如果 LivePortrait 官方模型对动物支持不好：

**方案 A: 使用社区训练的动物 checkpoint**
- 搜索 GitHub / HuggingFace: "LivePortrait animal checkpoint"
- 或查看 LivePortrait 官方 issue 中关于动物的讨论

**方案 B: 使用替代模型（SadTalker / AnimateDiff）**
- SadTalker: `https://github.com/OpenTalker/SadTalker` (对动物有一定支持)
- 生成视频后，同样用上面的步骤 5 提取帧并生成网格图

**方案 C: 使用 Replicate API（如果找到支持动物的模型）**
- 在 Colab 中调用 Replicate API，避免本地安装复杂依赖
- 参考下面的备用代码单元格


## 备用方案: 使用 Replicate API（如果本地模型不可用）


In [ ]:
# 如果 LivePortrait 本地运行遇到问题，可以尝试通过 Replicate API
# 注意：需要设置 REPLICATE_API_TOKEN 环境变量

# !pip install replicate
# import replicate
# import os
# 
# # 设置 API Token（从 Replicate 网站获取）
# os.environ["REPLICATE_API_TOKEN"] = "your_token_here"
# 
# # 上传图片和视频到可访问的 URL（或使用 Replicate 的文件上传）
# # 然后调用 API（参考之前的 app.py 代码）
# 
# model = replicate.models.get("fofr/live-portrait")
# latest_version = model.latest_version
# 
# prediction = replicate.predictions.create(
#     version=latest_version,
#     input={
#         "face_image": open("inputs/pet_source.jpg", "rb"),
#         "driving_video": open("inputs/driving_video.mp4", "rb"),
#     }
# )
# 
# # 等待完成并下载视频...
# # （然后继续步骤 5 提取帧）
